In [10]:
!pip install requests

In [12]:
"""
ETAPE 1 - Exploration des endpoints du guide phytosanitaire
Objectif : comprendre le format exact des reponses avant de construire le scraper final.
A lancer et me coller la SORTIE COMPLETE (les print) pour qu'on ajuste le parsing.
"""
import requests

BASE = "https://guidephytosanitaire.tn"

session = requests.Session()

headers = {
    "accept": "*/*",
    "accept-language": "fr-FR,fr;q=0.9",
    "content-type": "application/x-www-form-urlencoded; charset=UTF-8",
    "origin": BASE,
    "referer": f"{BASE}/index.php",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36",
    "x-requested-with": "XMLHttpRequest",
}

# 1) Charger la page d'accueil d'abord pour obtenir un PHPSESSID valide dans la session
r0 = session.get(f"{BASE}/index.php", headers={"user-agent": headers["user-agent"]})
print("=== index.php ===")
print("Status:", r0.status_code)
print("Cookies recus:", session.cookies.get_dict())
print()

# 2) Tester get_cible.php pour la culture "Olivier"
r1 = session.post(
    f"{BASE}/get_cible.php",
    headers=headers,
    data={"nature": "", "culture": "Olivier", "groupes": ""},
)
print("=== get_cible.php (culture=Olivier) ===")
print("Status:", r1.status_code)
print("Content-Type:", r1.headers.get("content-type"))
print("Reponse brute (500 premiers caracteres):")
print(r1.text[:500])
print()

# 3) Tester ajax_refresh.php pour l'autocomplete produit
r2 = session.post(
    f"{BASE}/ajax_refresh.php",
    headers=headers,
    data={"keyword": "tomate"},
)
print("=== ajax_refresh.php (keyword=tomate) ===")
print("Status:", r2.status_code)
print("Reponse brute (500 premiers caracteres):")
print(r2.text[:500])
print()

# 4) Tester rech1.php (page de recherche - pour voir si elle liste les cultures disponibles)
r3 = session.get(f"{BASE}/rech1.php?bio=", headers={"user-agent": headers["user-agent"]})
print("=== rech1.php?bio= ===")
print("Status:", r3.status_code)
print("Longueur HTML:", len(r3.text))
# On cherche les <option> pour lister les cultures possibles
import re
options = re.findall(r'<option[^>]*value="([^"]*)"[^>]*>([^<]*)</option>', r3.text)
print("Options trouvees (value, texte) - premieres 20:")
for opt in options[:20]:
    print(" ", opt)

=== index.php ===
Status: 200
Cookies recus: {}

=== get_cible.php (culture=Olivier) ===
Status: 200
Content-Type: text/html
Reponse brute (500 premiers caracteres):
<head>
        <meta http-equiv="Content-Type" content="text/html; charset=utf-8" />

</head>
	        <option value="">CIBLE</option>
           <optgroup   id="Olivier"  label="Olivier">

	<option value="Acariens">Acariens</option>
	<option value="Cochenille">Cochenille</option>
	<option value="Dacus">Dacus</option>
	<option value="HylÃ©sine">HylÃ©sine</option>
	<option value="Neiroun">Neiroun</option>
	<option value="Psylle">Psylle</option>
	<option value="Pyrale de jasmin">Pyral

=== ajax_refresh.php (keyword=tomate) ===
Status: 200
Reponse brute (500 premiers caracteres):


=== rech1.php?bio= ===
Status: 200
Longueur HTML: 1050356
Options trouvees (value, texte) - premieres 20:


In [13]:
"""
ETAPE 2 - Corriger l'encodage + trouver le select CULTURE + tester rech2.php
A lancer et me coller la sortie complete.
"""
import requests
from bs4 import BeautifulSoup

BASE = "https://guidephytosanitaire.tn"

session = requests.Session()

ua = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"

headers_ajax = {
    "accept": "*/*",
    "accept-language": "fr-FR,fr;q=0.9",
    "content-type": "application/x-www-form-urlencoded; charset=UTF-8",
    "origin": BASE,
    "referer": f"{BASE}/index.php",
    "user-agent": ua,
    "x-requested-with": "XMLHttpRequest",
}

# --- 1) get_cible.php avec encodage corrige ---
r1 = session.post(
    f"{BASE}/get_cible.php",
    headers=headers_ajax,
    data={"nature": "", "culture": "Olivier", "groupes": ""},
)
r1.encoding = "utf-8"  # forcer l'encodage correct
soup1 = BeautifulSoup(r1.text, "html.parser")
cibles = [opt["value"] for opt in soup1.find_all("option") if opt.get("value")]
print("=== CIBLES pour Olivier (encodage corrige) ===")
print(cibles)
print()

# --- 2) Chercher le select CULTURE dans index.php ---
r_index = session.get(f"{BASE}/index.php", headers={"user-agent": ua})
r_index.encoding = "utf-8"
soup_index = BeautifulSoup(r_index.text, "html.parser")

# On cherche tous les <select> de la page et leurs noms/id
selects = soup_index.find_all("select")
print("=== SELECTS trouves sur index.php ===")
for s in selects:
    print("name=", s.get("name"), "| id=", s.get("id"), "| nb options=", len(s.find_all("option")))
print()

# Si un select semble etre "culture", on affiche ses options
for s in selects:
    name_or_id = (s.get("name") or "") + (s.get("id") or "")
    if "ultur" in name_or_id.lower():
        opts = [opt.get("value") for opt in s.find_all("option") if opt.get("value")]
        print(f"=== Options du select '{name_or_id}' ===")
        print(opts)
        print()

# --- 3) Tester ajax_refresh avec un mot-cle plus probable (partiel produit) ---
r2 = session.post(f"{BASE}/ajax_refresh.php", headers=headers_ajax, data={"keyword": "gold"})
r2.encoding = "utf-8"
print("=== ajax_refresh.php (keyword=gold) ===")
print(repr(r2.text[:500]))
print()

# --- 4) Construire le parametre t serialise PHP et tester rech2.php ---
def php_serialize_t(nature="", champ1="", culture="", cible="", matiere="", societe="", fabricant=""):
    fields = [nature, champ1, culture, cible, matiere, societe, fabricant]
    parts = []
    for i, val in enumerate(fields):
        byte_len = len(val.encode("utf-8"))
        parts.append(f'i:{i};s:{byte_len}:"{val}";')
    return "a:7:{" + "".join(parts) + "}"

t_value = php_serialize_t(culture="Olivier")
print("=== t serialise construit ===")
print(t_value)
print()

r3 = session.get(
    f"{BASE}/rech2.php",
    params={"bio": "non", "t": t_value, "prod": ""},
    headers={"user-agent": ua, "referer": f"{BASE}/rech1.php?bio="},
)
r3.encoding = "utf-8"
print("=== rech2.php (culture=Olivier, sans autre filtre) ===")
print("Status:", r3.status_code)
print("Longueur:", len(r3.text))
print("Premiers 2000 caracteres:")
print(r3.text[:2000])

=== CIBLES pour Olivier (encodage corrige) ===
['Acariens', 'Cochenille', 'Dacus', 'Hylésine', 'Neiroun', 'Psylle', 'Pyrale de jasmin', 'Teigne', 'œil de paon', 'Dacus', 'Acariens', 'Carpocapse', 'Cératites', 'Mineuses', 'Mouche blanche', 'Mouches mineuses', 'Psylle', 'Pucerons', 'Scolytes', 'Teigne', 'Tordeuse', 'ver de jeunes fruits', 'Zeuzère', 'Bactériose', 'Chancre Cloque', 'Coryneum', 'Feu bactérien', 'Moniliose', 'Oïdium', 'Septoriose', 'Tavelure', "Traitement d'hiver", 'Traitement d’hiver', 'Chiendent', 'Cypérus', 'Diss', 'Mauvaises herbes annuelles', 'Mauvaises herbes vivaces', 'Cigales', 'Criquets pèlerins', 'Noctuelles', 'Chiendent', 'Cypérus', 'Diss', 'Escargot', 'Limace', 'Mouillant', 'Rat, mulot et souris domestiques', 'Rongeurs']

=== SELECTS trouves sur index.php ===
name= list-nature | id= list-nature | nb options= 5
name= list-groupes | id= list-groupes | nb options= 6
name= list-cultures | id= list-cultures | nb options= 90
name= list-cible | id= list-cible | nb opti